# PointNav + V-JEPA 2 experiment (JEPA-NAV baseline)

This notebook runs the same experiment as `scripts/train_pointnav_vjepa2.py`.

**Important:** Notebook vs script does **not** change whether Habitat-Sim has Bullet. The same Python env and same `habitat_sim` are used. If your env was installed without Bullet (e.g. conda/pip), ReplicaCAD will not load and `pathfinder.is_loaded` will be False here too.

Use this notebook to:
- Run interactively **after** Habitat-Sim with Bullet is installed (e.g. same env used by the install SLURM job).
- Inspect observations, embeddings, and metrics step by step.
- Debug or run a few episodes without submitting a SLURM job.

## 1. Project path and env check

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/ocean/projects/cis250225p/eajayi1/JEPANAV").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import habitat_sim
print("habitat_sim:", habitat_sim.__file__)
print("Python:", sys.executable)

## 2. Try loading ReplicaCAD and check navmesh (Bullet)

In [ ]:
# Uses same create_sim as the training script
from scripts.train_pointnav_vjepa2 import create_sim

sim = create_sim("apt_0")
pathfinder = sim.pathfinder
loaded = pathfinder.is_loaded
print("pathfinder.is_loaded:", loaded)
if not loaded:
    print("ReplicaCAD needs Habitat-Sim built WITH Bullet. Same as when you run the script.")
    print("Close the sim to avoid holding resources.")
sim.close()

## 3. Run training (only works if pathfinder was loaded above)

If cell 2 showed `pathfinder.is_loaded: True`, run this cell to train for a few episodes. Otherwise this will exit with the same "Navmesh not loaded" message.

In [ ]:
from scripts.train_pointnav_vjepa2 import train

train(
    scene_id="apt_0",
    num_episodes=10,
    max_steps=200,
    lr=3e-4,
    device_name="cuda",
    out_dir=PROJECT_ROOT / "checkpoints" / "pointnav_vjepa2",
    seed=42,
)

## 4. (Optional) Eval only with a checkpoint

In [ ]:
# Uncomment and set path after you have a checkpoint:
# from scripts.train_pointnav_vjepa2 import main
# import sys
# sys.argv = ["", "--eval_only", "--checkpoint", str(PROJECT_ROOT / "checkpoints/pointnav_vjepa2/policy_last.pt"), "--num_episodes", "5"]
# main()  # need to patch argparse or call eval logic directly
print("Use CLI: python scripts/train_pointnav_vjepa2.py --eval_only --checkpoint checkpoints/pointnav_vjepa2/policy_last.pt --num_episodes 5")